In [ ]:
import pandas as pd
import polars as pl

In [ ]:
def dms_to_decimal(dms):
    """Convert DMS packed as ±DDMMSS or ±DDDMMSS to decimal degrees."""
    sign = -1 if str(dms).startswith("-") else 1
    dms = abs(int(dms))

    degrees = dms // 10000
    minutes = (dms % 10000) // 100
    seconds = dms % 100

    val = round(sign * (degrees + minutes / 60 + seconds / 3600), 4)
    return val

In [ ]:
# Add decimal lat, lon and plot_id

df_plm = pl.read_csv("../data/raw/ICP/595_mm_20260227091917/mm_plm.csv", separator=";")

df_plm = df_plm.with_columns(
    [
        pl.col("latitude").map_elements(dms_to_decimal, return_dtype=pl.Float64).alias("Lat"),
        pl.col("longitude").map_elements(dms_to_decimal, return_dtype=pl.Float64).alias("Lon"),
    ]
)

df_plm = df_plm.with_columns(
    (
        pl.col("code_country").cast(pl.Utf8).str.zfill(2)
        + "."
        + pl.col("code_plot").cast(pl.Utf8).str.zfill(4)
    ).alias("plot_id")
)

df_plm.head()

In [ ]:
# -------------------------------------------------------------------
# Data loading and preprocessing for ICP MM data
#
# This cell performs the following steps:
#
# 1. Reads the raw ICP MM CSV file (semicolon-separated).
# 2. Perform a sanity check to ensure that mean value is between min and max.
# 3. Generates a unique `plot_id` by zero-padding and combining
#    `code_country` and `code_plot` into the format CC.PPPP.
# 4. Converts the `date_observation` column from string to `pl.Date`.
# 5. Extracts `year` and `month` from the observation date to support
#    temporal filtering and aggregation.
# 6. Creates a `month_year` column in MM-YYYY format for monthly aggregation.
# 7. Filters out historical observations from 1960 and earlier, which are
#    outside the scope of the analysis.
#
# The resulting DataFrame is prepared for downstream grouping,
# aggregation, and pivot operations.
# -------------------------------------------------------------------

df = pl.read_csv("../data/raw/ICP/595_mm_20260227091917/mm_mem.csv", separator=";")

df = df.with_columns(
    pl.col("daily_min").cast(pl.Float64),
    pl.col("daily_mean").cast(pl.Float64),
    pl.col("daily_max").cast(pl.Float64),
)

df = df.with_columns(
    [
        pl.when(pl.col("daily_mean") < pl.col("daily_min"))
        .then(None)
        .otherwise(pl.col("daily_min"))
        .alias("daily_min"),
        pl.when(pl.col("daily_mean") > pl.col("daily_max"))
        .then(None)
        .otherwise(pl.col("daily_max"))
        .alias("daily_max"),
    ]
)

df = (
    df.with_columns(
        (
            pl.col("code_country").cast(pl.Utf8).str.zfill(2)
            + "."
            + pl.col("code_plot").cast(pl.Utf8).str.zfill(4)
        ).alias("plot_id")
    )
    .with_columns(
        pl.col("date_observation").str.strptime(pl.Date, "%Y-%m-%d").alias("date_observation")
    )
    .with_columns(
        [
            pl.col("date_observation").dt.year().alias("year"),
            pl.col("date_observation").dt.month().alias("month"),
        ]
    )
    .with_columns(pl.col("date_observation").dt.strftime("%m-%Y").alias("month_year"))
    .filter(pl.col("year") > 1960)
)

df.head()

In [ ]:
df_ch = df.filter(pl.col("code_country") == 50)

df_ch.select("code_plot").unique().to_series().to_list()

In [ ]:
# There are duplicate observations for the same plot, variable, and date
# caused by multiple records with different `code_line` and `line_nr` values.
# These duplicates represent the same measurement context and should be
# consolidated into a single record.
#
# To resolve this, we group by the unique identifiers that define a single
# observation (country, plot, variable, date, plot_id, and month_year),
# and compute the mean of all remaining numeric columns across duplicates.
# This effectively averages over multiple entries for the same observation.
#
# After aggregation, we drop metadata and quality-control columns that are
# no longer meaningful once the duplicates have been collapsed.

df = (
    df.group_by(
        ["code_country", "code_plot", "code_variable", "date_observation", "plot_id", "month_year"]
    )
    .agg(
        [
            pl.all()
            .exclude(
                [
                    "code_country",
                    "code_plot",
                    "code_variable",
                    "date_observation",
                    "plot_id",
                    "month_year",
                ]
            )
            .mean()
        ]
    )
    .drop(
        [
            "code_data_origin",
            "code_data_status",
            "other_obs",
            "q_flag",
            "change_date",
            "code_line",
            "line_nr",
        ]
    )
)

df.head()

In [ ]:
df_pivoted = df.pivot(
    values=["daily_min", "daily_max", "daily_mean", "daily_completeness"],
    index=[
        "code_country",
        "code_plot",
        "plot_id",
        "month_year",
        "date_observation",
    ],
    on="code_variable",
)
df_pivoted = df_pivoted.select(
    [
        "code_country",
        "code_plot",
        "plot_id",
        "date_observation",
        "month_year",
        "daily_mean_PR",
        "daily_completeness_PR",
        "daily_mean_AT",
        "daily_min_AT",
        "daily_max_AT",
        "daily_completeness_AT",
        "daily_mean_RH",
        "daily_min_RH",
        "daily_max_RH",
        "daily_completeness_RH",
        "daily_mean_WS",
        "daily_max_WS",
        "daily_completeness_WS",
        "daily_mean_WD",
        "daily_completeness_WD",
        "daily_mean_SR",
        "daily_completeness_SR",
    ]
)
df_pivoted.head()

In [ ]:
# -------------------------------------------------------------------
# Variable-wise data completeness analysis

# This function analyzes data availability for a given variable
# (e.g., air temperature "AT") across plots and months.

# Execution steps:
# 1. Filter the input DataFrame to keep only rows corresponding to the
#    specified `code_variable`.
# 2. Report the number of unique month–year combinations available
#    for the selected variable, providing a quick overview of its
#    temporal coverage.
# 3. Group the filtered data by country, plot, and month–year.
# 4. For each group, check whether all values of `daily_mean`,
#    `daily_min`, and `daily_max` are missing (null).
# 5. Identify months where *all three* daily statistics are completely
#    null, indicating a full absence of usable data for that variable
#    in that plot and month.
# 6. Sort the resulting records chronologically by month–year.

# The function returns a DataFrame listing plot–month combinations
# where the selected variable has no valid observations at all.
# -------------------------------------------------------------------


def variable_wise_analysis(df: pl.DataFrame, variable: str):
    """Variable wise analysis for missing data."""
    tdf = df.filter(pl.col("code_variable") == variable)
    print(
        "Number of unique (month-year) data in ",
        variable,
        "is :",
        tdf.select(pl.col("month_year")).n_unique(),
    )
    null_months = (
        tdf.group_by(["code_country", "code_plot", "month_year"])
        .agg(
            [
                (pl.col("daily_mean").is_null().all()).alias("daily_mean_all_null"),
                (pl.col("daily_min").is_null().all()).alias("daily_min_all_null"),
                (pl.col("daily_max").is_null().all()).alias("daily_max_all_null"),
            ]
        )
        .filter(
            pl.col("daily_mean_all_null")
            & pl.col("daily_min_all_null")
            & pl.col("daily_max_all_null")
        )
    ).sort(by="month_year")
    return null_months


null_months = variable_wise_analysis(df, "AT")

null_months.head()

In [ ]:
# -------------------------------------------------------------------
# Monthly aggregation of daily climate variables

# This block computes monthly summary statistics for each plot and
# variable based on daily observations.

# Execution steps:
# 1. Replace NaN values in daily statistics with nulls to ensure
#    correct aggregation behavior in Polars.
# 2. Group data by plot, variable, and calendar month
#    (`plot_id`, `code_variable`, `month_year`, `year`, `month`).
# 3. Compute monthly aggregates:
#    - `avg_daily_mean`: mean of daily mean values within the month
#    - `min_daily_min`: mean of daily minimum values (monthly extreme)
#    - `max_daily_max`: mean of daily maximum values (monthly extreme)
#    - `frost_days`: count of days with minimum temperature below 0°C
# 4. Sort the resulting monthly summaries chronologically by year
#    and month.

# The resulting DataFrame provides a compact monthly representation
# of daily climate dynamics for each plot and variable.
# -------------------------------------------------------------------

monthly_summary = (
    df.with_columns(
        [
            pl.col("daily_mean").fill_nan(None),
            pl.col("daily_min").fill_nan(None),
            pl.col("daily_max").fill_nan(None),
        ]
    )
    .group_by(
        [
            "plot_id",
            "code_country",
            "code_plot",
            "code_variable",
            "month_year",
            "year",
            "month",
        ]
    )
    .agg(
        [
            pl.col("daily_mean").mean().alias("avg_daily_mean"),
            pl.col("daily_min").mean().alias("min_daily_min"),
            pl.col("daily_max").mean().alias("max_daily_max"),
            (pl.col("daily_min") < 0).sum().alias("frost_days"),
            pl.col("daily_completeness").mean().alias("avg_completeness"),
        ]
    )
    .sort(["year", "month"])
)

monthly_summary.head()

In [ ]:
monthly_summary.filter((pl.col("code_variable") == "SR") & (pl.col("plot_id") == "02.0008"))

In [ ]:
# -------------------------------------------------------------------
# Temporal coverage analysis for each plot and variable
#
# This block evaluates the time span and completeness of climate data
# for every plot-variable combination, identifying gaps in the
# monthly time series.
#
# Execution steps:
# 1. Convert `month_year` from string format (MM-YYYY) to a proper
#    date type to enable chronological calculations.
# 2. Group the monthly summary data by plot and variable
#    (`plot_id`, `code_variable`).
# 3. For each plot-variable group, compute temporal coverage metrics:
#    - `n_rows`: total number of months with actual data available
#    - `min_month_year`: earliest recorded month in the time series
#    - `max_month_year`: latest recorded month in the time series
#    - `n_months`: expected total months if data were continuous
#      from the first to last observation (span calculation)
# 4. Calculate the expected number of months by converting the
#    date range into a month count (inclusive of both endpoints).
# 5. Sort the results by plot ID for easier interpretation.
#
# The resulting DataFrame provides a quick diagnostic to assess:
# - Data continuity (n_rows vs n_months)
# - Temporal extent for each plot-variable combination
# - Which plots have full vs. fragmented time series
# -------------------------------------------------------------------

plot_temporal_coverage = (
    monthly_summary.with_columns(
        # Ensure month_year is a proper date
        pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")
    )
    .group_by("plot_id", "code_variable")
    .agg(
        [
            pl.len().alias("n_rows"),  # number of rows
            pl.col("month_year").min().alias("min_month_year"),  # earliest month
            pl.col("month_year").max().alias("max_month_year"),  # latest month
            (
                (pl.col("month_year").max().dt.year() - pl.col("month_year").min().dt.year()) * 12
                + (pl.col("month_year").max().dt.month() - pl.col("month_year").min().dt.month())
                + 1
            ).alias("n_months"),  # months span
        ]
    )
    .sort("plot_id")
)

plot_temporal_coverage.head()

In [ ]:
# -------------------------------------------------------------------
# Identification of missing months in plot-variable time series
#
# This block builds on the temporal coverage summary to explicitly
# list which months are missing from each plot's data record.
#
# Execution steps:
# 1. Filter the `plot_temporal_coverage` DataFrame to retain only
#    plots where actual rows (`n_rows`) do not equal expected months
#    (`n_months`), indicating incomplete data.
# 2. For each plot-variable combination, retrieve the full list of
#    existing month-year values from the monthly summary.
# 3. Generate a complete sequence of all months between the minimum
#    and maximum recorded months using `pd.date_range`.
# 4. Compare the existing months against the complete sequence to
#    identify missing months.
# 5. Collect the count of missing months and the list of missing
#    month-year values for each plot.
# 6. Join the missing months information back to the original
#    `plot_temporal_coverage` DataFrame.
#
# The resulting DataFrame adds `n_missing_months` and `missing_months`
# columns, enabling detailed gap analysis and data quality assessment.
# -------------------------------------------------------------------


def missing_months(min_date, max_date, existing_months):
    """Create all months between min and max months."""
    all_months = pd.date_range(start=min_date, end=max_date, freq="MS").strftime("%m-%Y").to_list()

    missing = [d for d in all_months if d not in existing_months]
    return missing


incomplete_plots = plot_temporal_coverage.filter(pl.col("n_rows") != pl.col("n_months"))

# Collect month lists for each plot
missing_months_list = []

for row in plot_temporal_coverage.iter_rows(named=True):
    plot_id = row["plot_id"]
    code_var = row["code_variable"]
    min_month = row["min_month_year"]
    max_month = row["max_month_year"]

    # Existing months for this plot
    existing_months = (
        monthly_summary.filter(
            (pl.col("plot_id") == plot_id) & (pl.col("code_variable") == code_var)
        )
        .select("month_year")
        .to_series()
        .to_list()
    )

    missing = missing_months(min_month, max_month, existing_months)

    missing_months_list.append(
        {
            "plot_id": plot_id,
            "code_variable": code_var,
            "n_missing_months": len(missing),
            "missing_months": missing,
        }
    )

# Convert to DataFrame
missing_months_df = pl.DataFrame(missing_months_list)

plot_temporal_coverage = plot_temporal_coverage.join(
    missing_months_df, on=["plot_id", "code_variable"]
)

plot_temporal_coverage.head()

In [ ]:
plot_temporal_coverage = plot_temporal_coverage.join(
    missing_months_df, on=["plot_id", "code_variable"]
)

plot_temporal_coverage.head()

In [ ]:
plot_temporal_coverage.filter(pl.col("n_missing_months") == 0)

In [ ]:
# -------------------------------------------------------------------
# Extraction and aggregation of sample plot data for climate simulation
#
# This block selects a single plot with high-quality, continuous data
# (plot_id = "02.0008") and prepares a consolidated monthly weather
# dataset suitable for climate analysis and simulation modeling.
#
# Execution steps:
# 1. Filter the main dataset to retain only the target plot
#    ("02.0008"), which was previously identified as having 360
#    consecutive months of complete data (no missing months).
#
# 2. Split the filtered data into separate DataFrames by variable type:
#    - `temp_df`: Air temperature data (code_variable = "AT")
#    - `prcp_df`: Precipitation data (code_variable = "PR")
#    - `srad_df`: Solar radiation data (code_variable = "SR")
#
# 3. Aggregate each variable to monthly resolution:
#    a. Temperature aggregation (`mtemp_df`):
#       - `tmp_ave`: mean daily temperature for the month
#       - `tmp_min`: minimum daily minimum temperature (monthly extreme)
#       - `tmp_max`: maximum daily maximum temperature (monthly extreme)
#       - `frost_days`: count of days with minimum temperature below 0°C
#
#    b. Precipitation aggregation (`mprcp_df`):
#       - `prcp`: sum of daily precipitation for the month
#
#    c. Solar radiation aggregation (`msrad_df`):
#       - `srad`: mean daily solar radiation for the month
#
# 4. Merge the three aggregated DataFrames on `month_year` to create
#    a unified monthly weather dataset.
#
# 5. Perform post-aggregation processing:
#    a. Convert `month_year` from string (MM-YYYY) to proper date type
#    b. Extract `year` and `month` components for temporal sorting
#    c. Select and reorder columns to a clean, logical structure:
#       [year, month, tmp_ave, tmp_min, tmp_max, frost_days, prcp, srad]
#    d. Sort chronologically by year and month
#    e. Drop any rows with null values to ensure data completeness
#
# 6. Convert the final Polars DataFrame to pandas format for potential
#    export or compatibility with other libraries (optional Excel export
#    is commented out).
#
# The resulting `weather_df` provides a clean, complete monthly climate
# time series for the selected plot, ready for:
# - Climate trend analysis
# - Crop growth modeling
# - Statistical simulation
# - Machine learning applications
# -------------------------------------------------------------------


# plot_id = "02.0008" has 360 months of consecutive data,
# so we select this plot for sample simulations

plot_df = df.filter(pl.col("plot_id") == "02.0008")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").mean().alias("tmp_min"),
    pl.col("daily_max").mean().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

# weather_df.to_excel("../data/intermediate/weather_data.xlsx", index=False)

weather_df

In [ ]:
plot_temporal_coverage.filter(pl.col("plot_id") == "50.0018")

In [ ]:
# Plot: Davos

plot_df = df.filter(pl.col("plot_id") == "50.0018")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").mean().alias("tmp_min"),
    pl.col("daily_max").mean().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
)

weather_df.to_pandas().to_excel("../data/intermediate/Davos_weather_data.xlsx", index=False)

weather_df.head()

In [ ]:
plot_df = df.filter(pl.col("plot_id") == "51.0001")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").mean().alias("tmp_min"),
    pl.col("daily_max").mean().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
)

# weather_df.to_pandas().to_excel("../data/intermediate/weather_data.xlsx", index=False)

weather_df.head()

In [ ]:
# -------------------------------------------------------------------
# Summary table for weather_df: completeness and gap analysis
#
# This block generates a comprehensive summary of the weather_df
# dataset, evaluating data completeness, identifying gaps, and
# assessing temporal continuity.
# -------------------------------------------------------------------

# Create a summary DataFrame for weather_df
summary_stats = {
    "Metric": [
        "Total records (months)",
        "Start date",
        "End date",
        "Expected months (based on date range)",
        "Actual months present",
        "Missing months",
        "Completeness (%)",
        "Has gaps",
        "Number of gaps",
        "Largest gap (months)",
        "Average gap size (months)",
        "Unique years",
        "Unique months",
    ],
    "Value": [],
}

# Calculate basic metrics
total_records = weather_df.height
start_date = (
    weather_df.select(pl.col("year").min()).item(),
    weather_df.select(pl.col("month").min()).item(),
)
end_date = (
    weather_df.select(pl.col("year").max()).item(),
    weather_df.select(pl.col("month").max()).item(),
)

# Calculate expected months if continuous
start_year, start_month = start_date
end_year, end_month = end_date
expected_months = (end_year - start_year) * 12 + (end_month - start_month) + 1

# Check for gaps
weather_pl = weather_df.with_columns(pl.date(pl.col("year"), pl.col("month"), 1).alias("date"))
min_date = weather_pl.select(pl.col("date").min()).item()
max_date = weather_pl.select(pl.col("date").max()).item()
all_months = pl.date_range(start=min_date, end=max_date, interval="1mo", eager=True)
existing_months = set(weather_pl.select("date").to_series().to_list())
miss_months = sorted(set(all_months) - existing_months)

# Calculate gap statistics
gaps = []
if missing_months:
    # Find consecutive missing months (gaps)
    missing_series = pl.Series(miss_months)
    if len(missing_series) > 0:
        diff = missing_series.diff().dt.total_days()
        gap_starts = missing_series.filter((diff.is_null()) | (diff > 31)).to_list()
        gap_lengths = []

        if len(gap_starts) > 0:
            for i, gap_start in enumerate(gap_starts):
                if i < len(gap_starts) - 1:
                    gap_end = gap_starts[i + 1]
                    gap_months = len(
                        missing_series.filter(
                            (missing_series >= gap_start) & (missing_series < gap_end)
                        )
                    )
                else:
                    gap_months = len(missing_series.filter(missing_series >= gap_start))
                gap_lengths.append(gap_months)
        else:
            gap_lengths = [len(missing_series)]

        gaps = gap_lengths
    else:
        gaps = [len(miss_months)]

# Count years with complete data
weather_df = weather_df.with_columns(
    (pl.col("year").cast(pl.Utf8) + "-" + pl.col("month").cast(pl.Utf8).str.zfill(2)).alias(
        "year_month"
    )
)
complete_years = weather_df.group_by("year").agg(pl.col("month").n_unique().alias("month_count"))
years_complete = (
    complete_years.filter(pl.col("month_count") == 12).select("year").to_series().to_list()
)

# Populate summary values
summary_stats["Value"].extend(
    [
        total_records,
        f"{start_date[0]}-{start_date[1]:02d}",
        f"{end_date[0]}-{end_date[1]:02d}",
        expected_months,
        total_records,
        len(miss_months),
        f"{(total_records / expected_months * 100):.1f}%",
        "Yes" if missing_months else "No",
        len(gaps),
        max(gaps) if gaps else 0,
        f"{sum(gaps) / len(gaps):.1f}" if gaps else "N/A",
        weather_df.select(pl.col("year").n_unique()).item(),
        weather_df.select(pl.col("month").n_unique()).item(),
    ]
)

if len(miss_months) > 0:
    print("The data is not complete. \n")


summary_df = pd.DataFrame(summary_stats)
print("WEATHER DATASET COMPLETENESS SUMMARY")
summary_df

In [ ]:
# Detailed missing months analysis

print("MISSING MONTHS ANALYSIS")

if miss_months:
    # Group missing months by year
    missing_df = pl.DataFrame({"missing_date": miss_months})
    missing_df = missing_df.with_columns(
        [
            pl.col("missing_date").dt.year().alias("year"),
            pl.col("missing_date").dt.month().alias("month"),
        ]
    )
    missing_by_year = missing_df.group_by("year").agg(pl.len()).sort("year")

    print("\nMissing months by year:")
    print(missing_by_year)

    print("\nFirst 10 missing months:")
    for month in miss_months[:10]:
        print(f"  - {month.strftime('%B %Y')}")

    if len(miss_months) > 10:
        print(f"  ... and {len(miss_months) - 10} more")

    # Check if missing months are at the beginning or end
    if miss_months[0] == all_months[0]:
        print("\n Missing data at the beginning of the time series")
    if miss_months[-1] == all_months[-1]:
        print("\n Missing data at the end of the time series")
else:
    print("\n No missing months detected. The dataset is perfectly continuous!")